## Versión final organizada

Este notebook contiene únicamente los experimentos CSE por lotes y su resumen. Usa las mismas rutas de `outputs/` que el experimento original, por lo que puede reanudar los resultados ya calculados.


In [ ]:
from project_paths import (
    PROJECT_ROOT, ANALYSIS_DATA_DIR, OSSE_DATA_DIR,
    cse_experiment_paths, ensure_output_directories,
)
ensure_output_directories()
EXPERIMENT_NAME = 'xyz_control_nature_qiwenevents'
EXPERIMENT_ROOT, RESULTS_DIR, EXPERIMENT_FIGURES_DIR = cse_experiment_paths(EXPERIMENT_NAME)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# Qiwen CSE xyz control — Batch experiments


Este notebook ejecuta de forma autónoma las realizaciones estadísticas del CSE:

1. barrido de 40 eventos con horizonte fijo;
2. rejilla completa T–D de la Figura 3(a);
3. resumen y comparación con los valores de Qiwen.

La implementación científica permanece centralizada en `class_cse_qiwen.py`. Los archivos existentes se cargan para continuar desde el primer evento pendiente; no se eliminan ni se reinician resultados completos.


## Imports


In [ ]:
import importlib
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from class_da_sys import da_system
from class_Lorenz63 import run_RK4
from class_cse import plot_controlled_vs_free_nature_run
import class_cse_qiwen as cse_qiwen
cse_qiwen = importlib.reload(cse_qiwen)
from class_cse_qiwen import (
    build_and_apply_perturbation,
    detect_regime_changes,
    find_past_safe_trajectory,
    integrate_and_align,
    lorenz63,
    plot_found_past_safe_trajectory,
    plot_cse_initialization,
    plot_failed_event_diagnostic,
    plot_3d_nature_run_comparison,
    plot_lorenz_attractor_aligned_cse_starts,
    plot_regime_crossings,
    run_single_cse as run_single_cse_core,
    run_single_cse_mini_test,
    summarize_cse_batch_results,
    summarize_cse_batch_results_by_D,
    run_perturbation_unit_test,
)


## DA System Setup


In [ ]:
BASE_PATH = PROJECT_ROOT
PathFigures = EXPERIMENT_FIGURES_DIR
PathFigures.mkdir(parents=True, exist_ok=True)

# This notebook assimilates x, y, z and applies CSE perturbations to x, y, and z.
# control_indices=None makes build_and_apply_perturbation skip component masking
# and perturb x, y, and z, which is the original CSE case in Figure 3(a) of the paper.
CONTROL_INDICES = None
CONTROL_LABEL = 'xyz'
REGIME_REFERENCE_MODE = 'nature'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

das = da_system().load(ANALYSIS_DATA_DIR / 'x_analysis_init.pkl')

sv = das.getStateVector()
x_nature = sv.getTrajectory()

obs = das.getObsData()
y_obs = obs.getVal()

Ta = das.acyc_step
fcst_dt = das.fcst_dt

das.multiplicative_inflation = 1.04


## Load OSSE Histories


In [ ]:
with open(OSSE_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as f:
    osse_Xa_history = pickle.load(f)

with open(OSSE_DATA_DIR / 'osse_summary.pkl', 'rb') as f:
    osse_summary = pickle.load(f)

print('osse_Xa_history shape:', osse_Xa_history.shape)
print("osse_summary['t0']:", osse_summary['t0'])
print("osse_summary['num_cycles']:", osse_summary['num_cycles'])


## Detect Regime Changes


In [ ]:
# Detect every regime transition in the NATURE trajectory as a sign change
# of the x component (a crossing of x=0 between two consecutive time steps).
# With EVENT_SELECTION='qiwen', the function accepts crossings in both
# directions, keeps those occurring after step 500 whose preceding residence
# time is longer than 385 steps, and retains at most the first 100 events.
# The complete list of crossings is also saved in RESULTS_DIR for reuse.
regime_change_info = detect_regime_changes(
    x_nature,
    RESULTS_DIR,
    position=position,
    output_filename='cse_qiwen_xyz_control_change_region_time_qiwen_qiwenevents.pkl',
    event_selection=EVENT_SELECTION,
)

# All detected x=0 crossings in the NATURE run.
change_region_time = regime_change_info['change_region_time']
# Forecast/control horizon used by the function's diagnostic sign check.
check_position = regime_change_info['check_position']
# Filtered crossing times used as the CSE event set.
t_start_series = regime_change_info['t_start_series']
# Time spent in the previous regime before each selected crossing.
t_start_residence = regime_change_info['t_start_residence']
# Mean interval between all detected crossings, before event filtering.
mean_crossing_interval = regime_change_info['mean_crossing_interval']


## Batch Parameters

Se conservan los mismos parámetros del experimento individual. `experiment_step` limita cada realización y `position` proporciona el horizonte fijo de 4T0 usado en el primer batch.


In [ ]:
# Keep the shared CSE defaults consistent with the individual-event notebook.
# The batch cells replace D and EVENT_INDEX with their own D_run and event index;
# experiment_step remains the maximum number of DA cycles per realization.
D = 0.05
amp = 1.0 / D
experiment_step = 1000
EVENT_INDEX = 1

# Lorenz-63 parameters and integration time step used by the CSE.
solver_params = {
    'model_func': lorenz63,
    'sigma': 10.0,
    'beta': 8.0 / 3.0,
    'rho': 28.0,
    'dt': fcst_dt,
}

print('position:', position)


## Reusable Single-Event Runner

Esta envoltura conecta la configuración del notebook con `run_single_cse_core`. No modifica ni reduce el resultado devuelto.


In [ ]:
# Thin notebook wrapper around the single authoritative implementation.
# The complete result dictionary is returned unchanged so batch pickle files
# retain scalar metrics, trajectories, assimilation histories, branch choices,
# observations, perturbation times, and ensemble forecast diagnostics.
def run_single_cse(event_index, D_run, seed, regime_reference='nature', candidate_pool='full', cle_target_output_dir=None):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        control_indices=CONTROL_INDICES,
        T_0=T_0,
        multiple_number=multiple_number,
        cle_target_output_dir=cle_target_output_dir,
    )


## Diagnostics Preserved

Cada nueva realización se guarda completa en su archivo pickle. Se conservan las métricas de éxito, las trayectorias controladas, RMSE, búsquedas de trayectorias seguras, observaciones, ramas de control, tiempos perturbados, historiales del ensemble y pronósticos por miembro. Los pickles creados anteriormente se reutilizan sin modificarlos.


## Batch Run: 40 CSE Events


In [ ]:
# Run 40 xyz-control CSE events for each perturbation amplitude D.
# Unlike the full T-D grid below, this cell keeps the warning horizon fixed
# at the notebook value `position` (configured above as multiple_number * T_0).
# Event i always uses seed 1000 + i, making interrupted and repeated runs
# reproducible. Each D has its own pickle, updated after every completed event.
# If that pickle already exists, execution resumes from the first missing event.
# These fixed-horizon files also serve as a legacy [4T0] fallback in the grid summary.
# Storage note: current results include full diagnostic histories, so pickles may be large.
import pickle
import time

D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
REGIME_REFERENCE = 'nature'
#CANDIDATE_POOL = 'unchanged_bkp'
CANDIDATE_POOL = 'full'
N_BATCH_EVENTS = 40

for D_run in D_VALUES:
    output_suffix = '' if (REGIME_REFERENCE == 'nature' and CANDIDATE_POOL == 'full') else f'_{REGIME_REFERENCE}_{CANDIDATE_POOL}'
    output_file = RESULTS_DIR / f'cse_qiwen_xyz_control_40_results_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'
    if output_file.exists():
        with open(output_file, 'rb') as f:
            results = pickle.load(f)
        print(f'Resuming {output_file}: {len(results)}/{N_BATCH_EVENTS} events already done')
    else:
        results = []

    for i in range(len(results), N_BATCH_EVENTS):
        t_event_start = time.time()
        results.append(run_single_cse(
            i,
            D_run,
            seed=1000 + i,
            regime_reference=REGIME_REFERENCE,
            candidate_pool=CANDIDATE_POOL,
        ))
        with open(output_file, 'wb') as f:
            pickle.dump(results, f)
        elapsed_min = (time.time() - t_event_start) / 60.0
        print(f'Saved partial result {i + 1}/{N_BATCH_EVENTS} for D={D_run} ({elapsed_min:.2f} min)')

    print('Saved:', output_file)


## Batch Run: Figure 3(a) T-D Grid


In [ ]:
# Reproduce the complete Figure 3(a) experiment for xyz control.
# Rows are five warning horizons T/T0; columns are nine perturbation amplitudes D.
# For every (T, D) combination, the first 40 detected events are evaluated with
# deterministic seeds (1000 + event_index): 5 * 9 * 40 = 1800 possible runs.
# `position_run` changes with T, while the DA system and all other CSE settings
# remain those configured earlier in the notebook. Results for each (T, D) pair
# are stored in a separate pickle and saved after every event, so the expensive
# calculation can resume safely without repeating completed events.
# Storage note: current results include full diagnostic histories, so a fresh grid may be large.
import pickle
import time

FIG3G_T_MULTIPLES = {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
}
FIG3G_D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
FIG3G_N_BATCH_EVENTS = 40
FIG3G_REGIME_REFERENCE = 'nature'
FIG3G_CANDIDATE_POOL = 'full'


def _fig3g_t_file_label(T_label):
    return (
        T_label.strip('[]')
        .replace('.', 'p')
        .replace('T0', 'T0')
    )


def _fig3g_output_file(T_label, D_run):
    T_file_label = _fig3g_t_file_label(T_label)
    output_suffix = (
        ''
        if (FIG3G_REGIME_REFERENCE == 'nature' and FIG3G_CANDIDATE_POOL == 'full')
        else f'_{FIG3G_REGIME_REFERENCE}_{FIG3G_CANDIDATE_POOL}'
    )
    return RESULTS_DIR / f'cse_qiwen_xyz_control_40_results_{T_file_label}_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'


def run_single_cse_fig3g(event_index, D_run, seed, position_run, T_label, multiple_run):
    row = run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position_run,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=FIG3G_N_BATCH_EVENTS,
        regime_reference=FIG3G_REGIME_REFERENCE,
        candidate_pool=FIG3G_CANDIDATE_POOL,
        control_indices=CONTROL_INDICES,
        T_0=T_0,
        multiple_number=multiple_run,
    )
    row['T_label'] = T_label
    row['multiple_number'] = float(multiple_run)
    row['position'] = int(position_run)
    return row


for T_label, multiple_run in FIG3G_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    print(f'=== T={T_label}, multiple={multiple_run}, position={position_run} ===')
    for D_run in FIG3G_D_VALUES:
        output_file = _fig3g_output_file(T_label, D_run)
        if output_file.exists():
            with open(output_file, 'rb') as f:
                results = pickle.load(f)
            print(f'Resuming {output_file.name}: {len(results)}/{FIG3G_N_BATCH_EVENTS} events already done')
        else:
            results = []

        for i in range(len(results), FIG3G_N_BATCH_EVENTS):
            t_event_start = time.time()
            results.append(run_single_cse_fig3g(
                event_index=i,
                D_run=D_run,
                seed=1000 + i,
                position_run=position_run,
                T_label=T_label,
                multiple_run=multiple_run,
            ))
            with open(output_file, 'wb') as f:
                pickle.dump(results, f)
            elapsed_min = (time.time() - t_event_start) / 60.0
            print(
                f'Saved partial result {i + 1}/{FIG3G_N_BATCH_EVENTS} '
                f'for T={T_label}, D={D_run} ({elapsed_min:.2f} min)'
            )

        print('Saved:', output_file)


## Figure 3(a) Grid Summary and Comparison


In [ ]:
# Post-process the Figure 3(a) T-D grid; this cell does not run new CSE experiments.
# It loads every available (T, D) pickle, creates event-level and aggregate tables,
# and calculates the 40-event success ratio for each grid point. If a dedicated
# [4T0] grid file is missing, the fixed-horizon Batch Run file is used as fallback.
# The resulting success matrix is compared with the transcribed Qiwen xyz-control
# values. Event tables, aggregate tables, both matrices, and the side-by-side
# Figure 3(a)-style plot are saved in the configured results/figures directories.
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = globals().get('BASE_PATH', Path.cwd())
PathFigures = globals().get('PathFigures', EXPERIMENT_FIGURES_DIR)
PathFigures.mkdir(parents=True, exist_ok=True)
T_0 = globals().get('T_0', 75.1)
OUTPUT_TAG = globals().get('OUTPUT_TAG', '_qiwenevents')

FIG3G_T_MULTIPLES = globals().get('FIG3G_T_MULTIPLES', {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
})
FIG3G_D_VALUES = globals().get('FIG3G_D_VALUES', [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
FIG3G_N_BATCH_EVENTS = globals().get('FIG3G_N_BATCH_EVENTS', 40)
FIG3G_REGIME_REFERENCE = globals().get('FIG3G_REGIME_REFERENCE', 'nature')
FIG3G_CANDIDATE_POOL = globals().get('FIG3G_CANDIDATE_POOL', 'full')

# Values transcribed from Figure 3(a) of the paper (control x,y,z), T×D grid.
QIWEN_FIG3A_CONTROL_XYZ_VALUES = globals().get('QIWEN_FIG3A_CONTROL_XYZ_VALUES', {
    '[1.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.0, 0.3: 0.0, 0.4: 0.0, 0.5: 0.025},
    '[2T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.025, 0.3: 0.0, 0.4: 0.0, 0.5: 0.0},
    '[2.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.025, 0.1: 0.250, 0.2: 0.275, 0.3: 0.250, 0.4: 0.200, 0.5: 0.175},
    '[3T0]': {0.02: 0.0, 0.03: 0.050, 0.04: 0.425, 0.05: 0.550, 0.1: 0.800, 0.2: 0.675, 0.3: 0.400, 0.4: 0.100, 0.5: 0.200},
    '[4T0]': {0.02: 0.050, 0.03: 0.975, 0.04: 0.925, 0.05: 0.975, 0.1: 0.825, 0.2: 0.825, 0.3: 0.725, 0.4: 0.500, 0.5: 0.525},
})


def _fig3g_t_file_label(T_label):
    return T_label.strip('[]').replace('.', 'p').replace('T0', 'T0')


def _fig3g_output_file(T_label, D_run):
    T_file_label = _fig3g_t_file_label(T_label)
    output_suffix = (
        ''
        if (FIG3G_REGIME_REFERENCE == 'nature' and FIG3G_CANDIDATE_POOL == 'full')
        else f'_{FIG3G_REGIME_REFERENCE}_{FIG3G_CANDIDATE_POOL}'
    )
    return RESULTS_DIR / f'cse_qiwen_xyz_control_40_results_{T_file_label}_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'


def _legacy_4T0_output_file(D_run):
    return RESULTS_DIR / f'cse_qiwen_xyz_control_40_results_D{D_run}{OUTPUT_TAG}.pkl'


event_rows = []
aggregate_rows = []
for T_label, multiple_run in FIG3G_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    for D_run in FIG3G_D_VALUES:
        result_file = _fig3g_output_file(T_label, D_run)
        if not result_file.exists() and T_label == '[4T0]':
            legacy_file = _legacy_4T0_output_file(D_run)
            if legacy_file.exists():
                result_file = legacy_file
        if not result_file.exists():
            print('Missing Figure 3(a) grid result file:', result_file)
            aggregate_rows.append({
                'T': T_label,
                'multiple_number': multiple_run,
                'position': position_run,
                'D': D_run,
                'n_events': 0,
                'success_ratio': np.nan,
                'mean_control_ratio': np.nan,
                'mean_false_alarm_ratio': np.nan,
                'mean_success_control_ratio': np.nan,
                'n_failures': np.nan,
                'result_file': str(result_file),
            })
            continue

        with open(result_file, 'rb') as f:
            rows = pickle.load(f)
        for row in rows:
            event_rows.append({
                'T': row.get('T_label', T_label),
                'multiple_number': row.get('multiple_number', multiple_run),
                'position': row.get('position', position_run),
                'D': D_run,
                'event': row['event_index'],
                'success': row['success'],
                'control_ratio': row['control_ratio'],
                'false_alarm_ratio': row['false_alarm_ratio'],
                'success_control_ratio': row['success_control_ratio'],
                'failure_cycle': row['failure_cycle'],
                'result_file': str(result_file),
            })
        aggregate_rows.append({
            'T': T_label,
            'multiple_number': multiple_run,
            'position': position_run,
            'D': D_run,
            'n_events': len(rows),
            'success_ratio': float(np.mean([row['success'] for row in rows])) if rows else np.nan,
            'mean_control_ratio': float(np.nanmean([row['control_ratio'] for row in rows])) if rows else np.nan,
            'mean_false_alarm_ratio': float(np.nanmean([row['false_alarm_ratio'] for row in rows])) if rows else np.nan,
            'mean_success_control_ratio': float(np.nanmean([row['success_control_ratio'] for row in rows])) if rows else np.nan,
            'n_failures': int(np.sum([not row['success'] for row in rows])) if rows else np.nan,
            'result_file': str(result_file),
        })

fig3g_event_table = pd.DataFrame(event_rows)
fig3g_aggregate_table = pd.DataFrame(aggregate_rows)
fig3g_success_matrix = fig3g_aggregate_table.pivot(index='T', columns='D', values='success_ratio')
fig3g_success_matrix = fig3g_success_matrix.reindex(index=list(FIG3G_T_MULTIPLES), columns=FIG3G_D_VALUES)

qiwen_fig3g_matrix = pd.DataFrame.from_dict(QIWEN_FIG3A_CONTROL_XYZ_VALUES, orient='index')
qiwen_fig3g_matrix = qiwen_fig3g_matrix.reindex(index=list(FIG3G_T_MULTIPLES), columns=FIG3G_D_VALUES)
fig3g_difference_matrix = fig3g_success_matrix - qiwen_fig3g_matrix

fig3g_event_table_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_grid_events_nature_qiwenevents.csv'
fig3g_aggregate_table_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_grid_aggregates_nature_qiwenevents.csv'
fig3g_success_matrix_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_success_matrix_nature_qiwenevents.csv'
fig3g_difference_matrix_file = RESULTS_DIR / 'cse_qiwen_xyz_control_40_fig3a_minus_qiwen_matrix_nature_qiwenevents.csv'
fig3g_event_table.to_csv(fig3g_event_table_file, index=False)
fig3g_aggregate_table.to_csv(fig3g_aggregate_table_file, index=False)
fig3g_success_matrix.to_csv(fig3g_success_matrix_file)
fig3g_difference_matrix.to_csv(fig3g_difference_matrix_file)

try:
    display(fig3g_success_matrix)
    display(fig3g_difference_matrix)
except NameError:
    print('This-run success matrix:')
    print(fig3g_success_matrix)
    print('This-run minus Qiwen matrix:')
    print(fig3g_difference_matrix)

PAPER_TABLE_COLORS = {
    'ge_08': '#9c7a00',
    'ge_06': '#d1a000',
    'ge_04': '#ffe173',
    'ge_02': '#fff0b3',
    'gt_00': '#fff8dc',
    'zero': '#ffffff',
    'missing': '#f2f2f2',
}


def _paper_cell_color(value):
    if pd.isna(value):
        return PAPER_TABLE_COLORS['missing']
    value = float(value)
    if np.isclose(value, 0.0):
        return PAPER_TABLE_COLORS['zero']
    if value >= 0.8:
        return PAPER_TABLE_COLORS['ge_08']
    if value >= 0.6:
        return PAPER_TABLE_COLORS['ge_06']
    if value >= 0.4:
        return PAPER_TABLE_COLORS['ge_04']
    if value >= 0.2:
        return PAPER_TABLE_COLORS['ge_02']
    return PAPER_TABLE_COLORS['gt_00']


def _format_paper_value(value):
    if pd.isna(value):
        return ''
    value = float(value)
    if np.isclose(value, 0.0):
        return '0'
    return f'{value:.3f}'.rstrip('0').rstrip('.')


def _format_D_label(value):
    return f'{float(value):.2f}'.rstrip('0').rstrip('.')


def _draw_paper_table(ax, matrix, title):
    ax.axis('off')
    header = [r'T \\ D'] + [_format_D_label(D_run) for D_run in matrix.columns]
    table_text = [header]
    table_colors = [['#ffffff'] * len(header)]

    for T_label in matrix.index:
        row_text = [T_label]
        row_colors = ['#ffffff']
        for D_run in matrix.columns:
            value = matrix.loc[T_label, D_run]
            row_text.append(_format_paper_value(value))
            row_colors.append(_paper_cell_color(value))
        table_text.append(row_text)
        table_colors.append(row_colors)

    mpl_table = ax.table(
        cellText=table_text,
        cellColours=table_colors,
        cellLoc='center',
        loc='center',
        bbox=[0.0, 0.0, 1.0, 0.88],
    )
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(8.4)
    mpl_table.scale(1.0, 1.16)

    for (row, col), table_cell in mpl_table.get_celld().items():
        table_cell.set_edgecolor('black')
        table_cell.set_linewidth(0.65)
        if row == 0 or col == 0:
            table_cell.set_text_props(weight='normal', color='black')
            table_cell.set_facecolor('#ffffff')
        elif table_cell.get_facecolor()[:3] == (0.611764705882353, 0.47843137254901963, 0.0):
            table_cell.set_text_props(color='white')

    ax.text(0.5, 0.945, title, ha='center', va='center', fontsize=10)
    return mpl_table


fig = plt.figure(figsize=(11.8, 4.5), constrained_layout=False)
grid = fig.add_gridspec(2, 2, height_ratios=[0.22, 1.0], hspace=0.12, wspace=0.08)
legend_ax = fig.add_subplot(grid[0, :])
legend_ax.axis('off')
legend_labels = [[r'$\geq 0.8$', r'$\geq 0.6$', r'$\geq 0.4$', r'$\geq 0.2$', r'$> 0$', r'$0$']]
legend_colors = [[
    PAPER_TABLE_COLORS['ge_08'],
    PAPER_TABLE_COLORS['ge_06'],
    PAPER_TABLE_COLORS['ge_04'],
    PAPER_TABLE_COLORS['ge_02'],
    PAPER_TABLE_COLORS['gt_00'],
    PAPER_TABLE_COLORS['zero'],
]]
legend_table = legend_ax.table(
    cellText=legend_labels,
    cellColours=legend_colors,
    cellLoc='center',
    loc='center',
    bbox=[0.38, 0.18, 0.24, 0.62],
)
legend_table.auto_set_font_size(False)
legend_table.set_fontsize(8.2)
for (row, col), table_cell in legend_table.get_celld().items():
    table_cell.set_edgecolor('black')
    table_cell.set_linewidth(0.65)
    if col == 0:
        table_cell.set_text_props(color='white')

axes = [fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])]
_draw_paper_table(axes[0], qiwen_fig3g_matrix, '(a) Qiwen control xyz')
_draw_paper_table(axes[1], fig3g_success_matrix, '(a) Diego control xyz')

fig3g_grid_png = PathFigures / 'cse_qiwen_xyz_control_fig3a_grid_comparison_nature_qiwenevents.png'
fig.savefig(fig3g_grid_png, dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Saved event table:', fig3g_event_table_file)
print('Saved aggregate table:', fig3g_aggregate_table_file)
print('Saved success matrix:', fig3g_success_matrix_file)
print('Saved difference matrix:', fig3g_difference_matrix_file)
print('Saved comparison figure:', fig3g_grid_png)
